# Manifestro Stage 1 Aether — LoquaciousSet Mimi cache

Run cells in order. The full extraction and Hugging Face publication are separate, explicit cells.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

CODE_REPO_URL = "https://github.com/karl4th/dataset-coll.git"
CODE_REVISION = "main"  # Replace with the published commit SHA before the full run.

drive.mount("/content/drive")
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "Add HF_TOKEN in Colab Secrets"
os.environ["HF_TOKEN"] = hf_token
os.environ["HF_HUB_CACHE"] = "/content/drive/MyDrive/.cache/huggingface/hub"
PROJECT_LOCAL = Path("/content/aether-dataset")
OUTPUT = Path("/content/drive/MyDrive/manifestro/stage1_aether_cache")
git_env = os.environ.copy()
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token:
    git_env["GITHUB_TOKEN"] = github_token
    askpass = Path("/content/git-askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        "*Username*) echo x-access-token ;;\n"
        "*) printf '%s' \"$GITHUB_TOKEN\" ;;\n"
        "esac\n"
    )
    askpass.chmod(0o700)
    git_env["GIT_ASKPASS"] = str(askpass)
    git_env["GIT_TERMINAL_PROMPT"] = "0"
if (PROJECT_LOCAL / ".git").is_dir():
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=PROJECT_LOCAL, text=True
    ).strip()
    assert origin == CODE_REPO_URL, f"Unexpected origin: {origin}"
    subprocess.run(
        ["git", "fetch", "--prune", "origin"], cwd=PROJECT_LOCAL, env=git_env, check=True
    )
else:
    assert not PROJECT_LOCAL.exists(), f"Non-Git path already exists: {PROJECT_LOCAL}"
    subprocess.run(["git", "clone", CODE_REPO_URL, str(PROJECT_LOCAL)], env=git_env, check=True)
subprocess.run(
    ["git", "checkout", "--detach", CODE_REVISION], cwd=PROJECT_LOCAL, env=git_env, check=True
)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_LOCAL, text=True).strip()
print("Checked out", commit)
assert (PROJECT_LOCAL / "pyproject.toml").is_file(), PROJECT_LOCAL
OUTPUT.mkdir(parents=True, exist_ok=True)
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
%pip install -q uv
subprocess.run(["uv", "sync", "--frozen", "--no-dev"], cwd=PROJECT_LOCAL, check=True)
config = PROJECT_LOCAL / "configs/loquacious-medium.yaml"
text = config.read_text()
text = text.replace("REQUIRED_OUTPUT_PATH", str(OUTPUT))
config.write_text(text)
subprocess.run(
    [
        "uv",
        "run",
        "--frozen",
        "python",
        "-c",
        "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))",
    ],
    cwd=PROJECT_LOCAL,
    check=True,
)

## Mandatory benchmark
Checks exact batch/single semantic-code equivalence and finds a safe GPU batch budget.

In [ ]:
subprocess.run(
    ["uv", "run", "--frozen", "aether-dataset", "benchmark", "--config", str(config)],
    cwd=PROJECT_LOCAL,
    check=True,
)

## Restartable full extraction
Safe to run again after a Colab reconnect.

In [ ]:
subprocess.run(
    ["uv", "run", "--frozen", "aether-dataset", "run", "--config", str(config), "--resume"],
    cwd=PROJECT_LOCAL,
    check=True,
)

In [ ]:
subprocess.run(
    ["uv", "run", "--frozen", "aether-dataset", "status", "--output", str(OUTPUT)],
    cwd=PROJECT_LOCAL,
    check=True,
)
subprocess.run(
    ["uv", "run", "--frozen", "aether-dataset", "validate", "--output", str(OUTPUT)],
    cwd=PROJECT_LOCAL,
    check=True,
)

## Explicit private publication
This cell writes to `manifestro/stage1_aether`. It refuses public repositories and incomplete caches.

In [ ]:
subprocess.run(
    [
        "uv",
        "run",
        "--frozen",
        "aether-dataset",
        "publish",
        "--output",
        str(OUTPUT),
        "--repo",
        "manifestro/stage1_aether",
    ],
    cwd=PROJECT_LOCAL,
    check=True,
)